# ACDADA — Notebook 07: Threat Intelligence Memory

**Vector Store for Attacker Behavior Patterns & CTI Knowledge**

This notebook implements:
1. Text preprocessing for CTI (Cyber Threat Intelligence) data
2. Embedding generation using sentence-transformers
3. FAISS vector index for fast similarity search
4. ChromaDB persistent collection as alternative
5. Threat pattern retrieval and enrichment pipeline
6. Integration helpers for the orchestration agent

In [ ]:
# ============================================================
# DATASET LINKS
# ============================================================
#
# Text-Based Cyber Threat Detection Dataset:
#   https://www.kaggle.com/datasets/ramoliyafenil/text-based-cyber-threat-detection
#
# SmartSys CTI — Anomaly Detection & Threat Intelligence:
#   https://www.kaggle.com/datasets/ziya07/anomaly-detection-and-threat-intelligence-dataset
#
# IoT Blockchain Security Dataset:
#   https://www.kaggle.com/datasets/ziya07/iot-blockchain-security-dataset
#
# Download and place in: ../data/raw/cti/
# ============================================================

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import json
import re
import hashlib
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RAW_DIR = Path('../data/raw/cti')
PROCESSED_DIR = Path('../data/processed/cti')
MODELS_DIR = Path('../models/threat_intel')

for d in [RAW_DIR, PROCESSED_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')

---
## 2. CTI Data Loading & Preprocessing

In [ ]:
@dataclass
class ThreatRecord:
    """Structured threat intelligence record."""
    record_id: str
    text: str
    category: str = 'unknown'
    severity: str = 'medium'
    source: str = 'unknown'
    tags: List[str] = field(default_factory=list)
    embedding: Optional[np.ndarray] = None
    metadata: Dict[str, Any] = field(default_factory=dict)


class CTIDataLoader:
    """Load and preprocess various CTI datasets."""
    
    # Known attack pattern keywords for tagging
    ATTACK_KEYWORDS = {
        'ddos': ['ddos', 'denial of service', 'flood', 'amplification', 'syn flood'],
        'malware': ['malware', 'trojan', 'ransomware', 'worm', 'virus', 'backdoor', 'rootkit'],
        'phishing': ['phishing', 'spear phishing', 'social engineering', 'credential'],
        'brute_force': ['brute force', 'password spray', 'credential stuffing', 'dictionary attack'],
        'injection': ['sql injection', 'sqli', 'xss', 'cross-site', 'code injection', 'command injection'],
        'apt': ['apt', 'advanced persistent', 'nation state', 'targeted attack'],
        'botnet': ['botnet', 'bot net', 'c2', 'command and control', 'c&c'],
        'exploitation': ['exploit', 'vulnerability', 'cve', 'zero-day', '0-day', 'rce'],
        'lateral_movement': ['lateral movement', 'pivot', 'pass the hash', 'pass the ticket'],
        'exfiltration': ['exfiltration', 'data theft', 'data leak', 'data breach'],
    }
    
    SEVERITY_KEYWORDS = {
        'critical': ['critical', 'severe', 'emergency', 'zero-day', 'rce', 'nation state'],
        'high': ['high', 'significant', 'dangerous', 'ransomware', 'apt'],
        'medium': ['medium', 'moderate', 'suspicious'],
        'low': ['low', 'minor', 'informational', 'benign'],
    }
    
    @staticmethod
    def clean_text(text: str) -> str:
        """Clean and normalize CTI text."""
        if not isinstance(text, str):
            return ''
        text = re.sub(r'https?://\S+', '[URL]', text)
        text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', '[IP]', text)
        text = re.sub(r'[a-fA-F0-9]{32,}', '[HASH]', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    @staticmethod
    def auto_tag(text: str) -> Tuple[List[str], str]:
        """Automatically tag and assign severity."""
        text_lower = text.lower()
        tags = []
        
        for category, keywords in CTIDataLoader.ATTACK_KEYWORDS.items():
            if any(kw in text_lower for kw in keywords):
                tags.append(category)
        
        severity = 'medium'
        for sev, keywords in CTIDataLoader.SEVERITY_KEYWORDS.items():
            if any(kw in text_lower for kw in keywords):
                severity = sev
                break
        
        return tags if tags else ['unknown'], severity
    
    @staticmethod
    def generate_id(text: str) -> str:
        return hashlib.md5(text.encode()).hexdigest()[:12]
    
    def load_text_cti(self, filepath: Path) -> List[ThreatRecord]:
        """Load text-based CTI dataset."""
        records = []
        try:
            df = pd.read_csv(filepath)
            print(f'Loaded {len(df)} rows from {filepath.name}')
            print(f'Columns: {list(df.columns)}')
            
            text_col = None
            label_col = None
            for col in df.columns:
                if 'text' in col.lower() or 'description' in col.lower() or 'content' in col.lower():
                    text_col = col
                if 'label' in col.lower() or 'class' in col.lower() or 'category' in col.lower() or 'type' in col.lower():
                    label_col = col
            
            if text_col is None:
                text_col = df.columns[0]
            
            for _, row in df.iterrows():
                text = self.clean_text(str(row[text_col]))
                if len(text) < 10:
                    continue
                
                category = str(row[label_col]).lower() if label_col and pd.notna(row[label_col]) else 'unknown'
                tags, severity = self.auto_tag(text)
                
                records.append(ThreatRecord(
                    record_id=self.generate_id(text),
                    text=text,
                    category=category,
                    severity=severity,
                    source='text_cti_dataset',
                    tags=tags,
                ))
            
            print(f'Created {len(records)} threat records.')
        except FileNotFoundError:
            print(f'File not found: {filepath}. Will use synthetic data.')
        
        return records


loader = CTIDataLoader()

# Try loading real data
cti_files = list(RAW_DIR.glob('*.csv'))
all_records = []
for f in cti_files:
    records = loader.load_text_cti(f)
    all_records.extend(records)

print(f'\nTotal records from disk: {len(all_records)}')

In [ ]:
# Generate synthetic CTI data if no real data available
SYNTHETIC_CTI = [
    # DDoS
    ('A volumetric DDoS attack was observed targeting port 80 with SYN flood reaching 50Gbps from botnet infrastructure.', 'ddos', 'high'),
    ('DNS amplification attack leveraging open resolvers to generate 100x traffic amplification against victim infrastructure.', 'ddos', 'high'),
    ('Application-layer DDoS targeting HTTP endpoints with randomized headers to bypass rate limiting defenses.', 'ddos', 'medium'),
    ('Distributed denial of service attack using NTP amplification with peak volume of 200Gbps sustained for 6 hours.', 'ddos', 'critical'),
    ('Memcached reflection DDoS campaign exploiting UDP port 11211 for amplification factors exceeding 50000x.', 'ddos', 'critical'),
    
    # Malware
    ('New ransomware variant discovered encrypting files with AES-256 and demanding Bitcoin payment via Tor hidden service.', 'malware', 'critical'),
    ('Trojan dropper distributed via phishing emails deploys keylogger and screen capture modules on infected systems.', 'malware', 'high'),
    ('Fileless malware utilizing PowerShell and WMI for persistence avoids traditional antivirus detection mechanisms.', 'malware', 'high'),
    ('Remote access trojan with rootkit capabilities detected establishing encrypted C2 channel to [IP] on port 443.', 'malware', 'critical'),
    ('Polymorphic worm spreading through SMB exploitation targeting unpatched Windows systems in enterprise networks.', 'malware', 'high'),
    
    # Phishing
    ('Spear phishing campaign targeting finance department employees with fake invoice attachments containing macro malware.', 'phishing', 'high'),
    ('Credential harvesting phishing site mimicking Office 365 login page hosted on compromised WordPress installation.', 'phishing', 'medium'),
    ('Business email compromise attack impersonating CEO requesting urgent wire transfer to offshore account.', 'phishing', 'high'),
    ('SMS phishing campaign distributing links to fake banking app designed to steal credentials and intercept MFA codes.', 'phishing', 'high'),
    ('Phishing emails with QR codes redirecting to credential harvesting pages to bypass email security gateways.', 'phishing', 'medium'),
    
    # Brute Force
    ('Distributed password spray attack targeting Azure AD accounts using rotated IP addresses from cloud infrastructure.', 'brute_force', 'high'),
    ('SSH brute force attempts from [IP] with 10000 login attempts per hour using common username and password combinations.', 'brute_force', 'medium'),
    ('Credential stuffing attack using leaked database of 2 million username password pairs against corporate VPN portal.', 'brute_force', 'high'),
    ('RDP brute force campaign targeting exposed Windows servers with dictionary attack on administrator accounts.', 'brute_force', 'medium'),
    ('Automated password spraying attack against IMAP email service targeting top 100 most common passwords per account.', 'brute_force', 'medium'),
    
    # Injection
    ('SQL injection vulnerability discovered in login form allowing authentication bypass and database extraction.', 'injection', 'critical'),
    ('Stored XSS vulnerability in web application comment section enables session hijacking of authenticated users.', 'injection', 'high'),
    ('Remote code execution via command injection in web application file upload functionality using OS command chaining.', 'injection', 'critical'),
    ('LDAP injection attack targeting Active Directory authentication module to enumerate user accounts and group memberships.', 'injection', 'high'),
    ('Server-side template injection allowing arbitrary code execution on Python Flask application backend server.', 'injection', 'critical'),
    
    # APT
    ('APT group attributed to nation state conducting long-term surveillance campaign targeting defense sector organizations.', 'apt', 'critical'),
    ('Advanced persistent threat actor maintaining access to compromised network for 18 months using custom implant framework.', 'apt', 'critical'),
    ('State-sponsored group using supply chain compromise to distribute backdoored software update to government agencies.', 'apt', 'critical'),
    ('Targeted attack using spear phishing with zero-day exploit in document processing application for initial access.', 'apt', 'critical'),
    ('APT campaign leveraging compromised VPN appliances as initial access vector into critical infrastructure networks.', 'apt', 'critical'),
    
    # Botnet
    ('IoT botnet compromising vulnerable CCTV cameras and routers using default credentials for DDoS-for-hire service.', 'botnet', 'high'),
    ('Peer-to-peer botnet architecture with encrypted C2 communication resistant to traditional sinkholing techniques.', 'botnet', 'high'),
    ('New botnet variant targeting Linux servers through exploitation of Log4Shell vulnerability for cryptomining operations.', 'botnet', 'high'),
    ('Command and control infrastructure identified using fast-flux DNS with hundreds of IP addresses rotating every 5 minutes.', 'botnet', 'medium'),
    ('Botnet establishing persistence through scheduled tasks and registry modifications for long-term command and control access.', 'botnet', 'high'),
    
    # Lateral Movement
    ('Attacker performing lateral movement using stolen Kerberos tickets via pass-the-ticket technique across domain controllers.', 'lateral_movement', 'critical'),
    ('Lateral movement detected using PsExec and Windows Management Instrumentation for remote code execution on workstations.', 'lateral_movement', 'high'),
    ('Internal reconnaissance and lateral movement using BloodHound to map Active Directory attack paths to domain admin access.', 'lateral_movement', 'high'),
    ('Pivot through compromised jump server to access segmented network containing sensitive database and file servers.', 'lateral_movement', 'critical'),
    ('Lateral movement using SSH key chain discovered on compromised Linux web server to access internal application servers.', 'lateral_movement', 'high'),
    
    # Exfiltration
    ('Data exfiltration detected via DNS tunneling with encoded payloads in subdomain queries to attacker-controlled nameserver.', 'exfiltration', 'critical'),
    ('Sensitive data being exfiltrated through HTTPS POST requests to cloud storage service blending with legitimate traffic.', 'exfiltration', 'high'),
    ('Data breach involving extraction of 500000 customer records through compromised API endpoint with no rate limiting.', 'exfiltration', 'critical'),
    ('Covert data exfiltration using steganography to embed stolen information in image files uploaded to social media platform.', 'exfiltration', 'high'),
    ('Database dump exfiltration through ICMP tunnel bypassing network segmentation and firewall egress filtering rules.', 'exfiltration', 'critical'),
    
    # Reconnaissance
    ('Network scanning activity detected from external IP targeting common service ports 22 80 443 3389 8080 across entire subnet.', 'reconnaissance', 'low'),
    ('Web application vulnerability scanning using automated tools Nikto and OWASP ZAP against public-facing web servers.', 'reconnaissance', 'medium'),
    ('DNS enumeration and zone transfer attempts targeting corporate domain to discover internal hostnames and IP addresses.', 'reconnaissance', 'medium'),
    ('Social media OSINT reconnaissance campaign gathering employee information for targeted phishing attack preparation.', 'reconnaissance', 'low'),
    ('Port scanning followed by service version fingerprinting indicating pre-attack reconnaissance of critical infrastructure.', 'reconnaissance', 'medium'),
]

if len(all_records) < 20:
    print('Insufficient real data. Generating synthetic CTI records...')
    for text, category, severity in SYNTHETIC_CTI:
        tags, _ = loader.auto_tag(text)
        all_records.append(ThreatRecord(
            record_id=loader.generate_id(text),
            text=text,
            category=category,
            severity=severity,
            source='synthetic',
            tags=tags,
        ))
    print(f'Total records after synthetic: {len(all_records)}')

# Show category distribution
categories = Counter(r.category for r in all_records)
print(f'\nCategories: {dict(categories)}')
severities = Counter(r.severity for r in all_records)
print(f'Severities: {dict(severities)}')

---
## 3. Embedding Generation

In [ ]:
class CTIEmbedder:
    """
    Generate embeddings for CTI text using sentence-transformers.
    Falls back to TF-IDF + SVD if sentence-transformers unavailable.
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self.fallback = False
        self.embed_dim = None
        
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name)
            self.embed_dim = self.model.get_sentence_embedding_dimension()
            print(f'Loaded sentence-transformer: {model_name} (dim={self.embed_dim})')
        except ImportError:
            print('sentence-transformers not available. Using TF-IDF + SVD fallback.')
            self.fallback = True
            self.embed_dim = 128
    
    def embed_texts(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if not self.fallback:
            embeddings = self.model.encode(
                texts, batch_size=batch_size,
                show_progress_bar=True, normalize_embeddings=True
            )
            return embeddings.astype(np.float32)
        else:
            return self._tfidf_embed(texts)
    
    def _tfidf_embed(self, texts: List[str]) -> np.ndarray:
        """Fallback: TF-IDF + Truncated SVD."""
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        from sklearn.preprocessing import normalize
        
        vectorizer = TfidfVectorizer(max_features=5000, stop_words='english',
                                     ngram_range=(1, 2), sublinear_tf=True)
        tfidf_matrix = vectorizer.fit_transform(texts)
        
        svd = TruncatedSVD(n_components=self.embed_dim, random_state=42)
        embeddings = svd.fit_transform(tfidf_matrix)
        embeddings = normalize(embeddings, norm='l2')
        
        # Save for inference
        import joblib
        joblib.dump(vectorizer, MODELS_DIR / 'tfidf_vectorizer.joblib')
        joblib.dump(svd, MODELS_DIR / 'tfidf_svd.joblib')
        print(f'TF-IDF SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}')
        
        return embeddings.astype(np.float32)
    
    def embed_query(self, query: str) -> np.ndarray:
        """Embed a single query text."""
        if not self.fallback:
            return self.model.encode([query], normalize_embeddings=True).astype(np.float32)[0]
        else:
            import joblib
            from sklearn.preprocessing import normalize
            vectorizer = joblib.load(MODELS_DIR / 'tfidf_vectorizer.joblib')
            svd = joblib.load(MODELS_DIR / 'tfidf_svd.joblib')
            tfidf = vectorizer.transform([query])
            emb = svd.transform(tfidf)
            return normalize(emb, norm='l2').astype(np.float32)[0]


# Generate embeddings
embedder = CTIEmbedder()
texts = [r.text for r in all_records]
embeddings = embedder.embed_texts(texts)

# Attach to records
for record, emb in zip(all_records, embeddings):
    record.embedding = emb

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'Embedding dim: {embedder.embed_dim}')

---
## 4. FAISS Vector Index

In [ ]:
import faiss


class FAISSThreatMemory:
    """
    FAISS-based vector store for threat intelligence.
    Supports:
    - Fast approximate nearest neighbor search
    - Metadata filtering
    - Incremental updates
    - Persistence
    """
    
    def __init__(self, embed_dim: int, index_type: str = 'flat'):
        self.embed_dim = embed_dim
        self.records: Dict[str, ThreatRecord] = {}
        self.id_to_idx: Dict[str, int] = {}
        self.idx_to_id: Dict[int, str] = {}
        
        # Build index
        if index_type == 'flat':
            # Exact search — best for < 100k vectors
            self.index = faiss.IndexFlatIP(embed_dim)  # Inner product (cosine for normalized vectors)
        elif index_type == 'ivf':
            # IVF for larger collections
            quantizer = faiss.IndexFlatIP(embed_dim)
            self.index = faiss.IndexIVFFlat(quantizer, embed_dim, 16, faiss.METRIC_INNER_PRODUCT)
        elif index_type == 'hnsw':
            # HNSW — best recall/speed tradeoff
            self.index = faiss.IndexHNSWFlat(embed_dim, 32, faiss.METRIC_INNER_PRODUCT)
        
        self.current_idx = 0
        print(f'FAISS index created: type={index_type}, dim={embed_dim}')
    
    def add_records(self, records: List[ThreatRecord]):
        """Add threat records to the index."""
        embeddings = []
        for record in records:
            if record.embedding is None:
                continue
            self.records[record.record_id] = record
            self.id_to_idx[record.record_id] = self.current_idx
            self.idx_to_id[self.current_idx] = record.record_id
            embeddings.append(record.embedding)
            self.current_idx += 1
        
        if embeddings:
            emb_matrix = np.vstack(embeddings).astype(np.float32)
            faiss.normalize_L2(emb_matrix)
            
            # Train IVF index if needed
            if hasattr(self.index, 'is_trained') and not self.index.is_trained:
                self.index.train(emb_matrix)
            
            self.index.add(emb_matrix)
        
        print(f'Added {len(embeddings)} vectors. Total: {self.index.ntotal}')
    
    def search(self, query_embedding: np.ndarray, k: int = 5,
               category_filter: Optional[str] = None,
               severity_filter: Optional[str] = None) -> List[Tuple[ThreatRecord, float]]:
        """Search for similar threat records."""
        query = query_embedding.reshape(1, -1).astype(np.float32)
        faiss.normalize_L2(query)
        
        # Over-fetch if filtering
        fetch_k = k * 5 if (category_filter or severity_filter) else k
        fetch_k = min(fetch_k, self.index.ntotal)
        
        scores, indices = self.index.search(query, fetch_k)
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            record_id = self.idx_to_id.get(idx)
            if record_id is None:
                continue
            record = self.records[record_id]
            
            # Apply filters
            if category_filter and record.category != category_filter:
                continue
            if severity_filter and record.severity != severity_filter:
                continue
            
            results.append((record, float(score)))
            if len(results) >= k:
                break
        
        return results
    
    def search_by_text(self, query_text: str, embedder: CTIEmbedder,
                       k: int = 5, **filters) -> List[Tuple[ThreatRecord, float]]:
        """Search using raw text query."""
        query_emb = embedder.embed_query(query_text)
        return self.search(query_emb, k=k, **filters)
    
    def save(self, path: Path):
        """Save index and metadata."""
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        
        faiss.write_index(self.index, str(path / 'faiss_index.bin'))
        
        metadata = {
            'embed_dim': self.embed_dim,
            'n_records': len(self.records),
            'id_to_idx': self.id_to_idx,
            'records': {
                rid: {
                    'text': r.text, 'category': r.category,
                    'severity': r.severity, 'source': r.source,
                    'tags': r.tags,
                } for rid, r in self.records.items()
            }
        }
        with open(path / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f'FAISS index saved to {path}')
    
    @classmethod
    def load(cls, path: Path, embedder: Optional[CTIEmbedder] = None) -> 'FAISSThreatMemory':
        """Load index and metadata."""
        path = Path(path)
        
        with open(path / 'metadata.json', 'r') as f:
            metadata = json.load(f)
        
        memory = cls(metadata['embed_dim'])
        memory.index = faiss.read_index(str(path / 'faiss_index.bin'))
        memory.id_to_idx = metadata['id_to_idx']
        memory.idx_to_id = {int(v): k for k, v in metadata['id_to_idx'].items()}
        memory.current_idx = memory.index.ntotal
        
        for rid, rdata in metadata['records'].items():
            memory.records[rid] = ThreatRecord(
                record_id=rid, text=rdata['text'],
                category=rdata['category'], severity=rdata['severity'],
                source=rdata['source'], tags=rdata['tags'],
            )
        
        print(f'Loaded FAISS index: {memory.index.ntotal} vectors')
        return memory


# Build FAISS index
faiss_memory = FAISSThreatMemory(embedder.embed_dim, index_type='flat')
faiss_memory.add_records(all_records)

In [ ]:
# Test FAISS search
test_queries = [
    'ransomware attack encrypting files and demanding payment',
    'DDoS flooding attack targeting web server',
    'attacker moving laterally through the network using stolen credentials',
    'SQL injection vulnerability in web application',
    'data exfiltration through encrypted DNS queries',
]

for query in test_queries:
    print(f'\n🔍 Query: "{query[:60]}..."')
    results = faiss_memory.search_by_text(query, embedder, k=3)
    for record, score in results:
        print(f'  [{score:.4f}] [{record.severity:8s}] [{record.category:18s}] {record.text[:80]}...')

---
## 5. ChromaDB Alternative

In [ ]:
class ChromaThreatMemory:
    """
    ChromaDB-based vector store.
    Offers built-in metadata filtering and persistence.
    """
    
    def __init__(self, collection_name='threat_intel', persist_dir=None):
        try:
            import chromadb
            from chromadb.config import Settings
            
            if persist_dir:
                self.client = chromadb.PersistentClient(path=str(persist_dir))
            else:
                self.client = chromadb.Client()
            
            # Delete existing collection if exists
            try:
                self.client.delete_collection(collection_name)
            except:
                pass
            
            self.collection = self.client.create_collection(
                name=collection_name,
                metadata={'hnsw:space': 'cosine'},
            )
            self.available = True
            print(f'ChromaDB collection "{collection_name}" created.')
        except ImportError:
            print('ChromaDB not installed. Using FAISS only.')
            self.available = False
    
    def add_records(self, records: List[ThreatRecord]):
        """Add records in batches."""
        if not self.available:
            return
        
        batch_size = 100
        for i in range(0, len(records), batch_size):
            batch = records[i:i+batch_size]
            ids = [r.record_id for r in batch]
            docs = [r.text for r in batch]
            embeds = [r.embedding.tolist() for r in batch if r.embedding is not None]
            metas = [{
                'category': r.category,
                'severity': r.severity,
                'source': r.source,
                'tags': ','.join(r.tags),
            } for r in batch]
            
            self.collection.add(
                ids=ids, documents=docs,
                embeddings=embeds[:len(ids)],
                metadatas=metas,
            )
        
        print(f'Added {len(records)} records to ChromaDB.')
    
    def search(self, query_embedding: np.ndarray, k: int = 5,
               where_filter: Optional[dict] = None) -> List[dict]:
        """Search with optional metadata filtering."""
        if not self.available:
            return []
        
        kwargs = {
            'query_embeddings': [query_embedding.tolist()],
            'n_results': k,
        }
        if where_filter:
            kwargs['where'] = where_filter
        
        results = self.collection.query(**kwargs)
        
        output = []
        for i in range(len(results['ids'][0])):
            output.append({
                'id': results['ids'][0][i],
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if results.get('distances') else None,
            })
        
        return output


# Try ChromaDB
chroma_memory = ChromaThreatMemory(
    collection_name='acdada_threat_intel',
    persist_dir=str(MODELS_DIR / 'chromadb'),
)
if chroma_memory.available:
    chroma_memory.add_records(all_records)

In [ ]:
# Test ChromaDB search with metadata filtering
if chroma_memory.available:
    query_emb = embedder.embed_query('ransomware encrypting critical files')
    
    print('=== All results ===')
    results = chroma_memory.search(query_emb, k=3)
    for r in results:
        print(f'  [{r["metadata"]["severity"]:8s}] [{r["metadata"]["category"]:18s}] {r["text"][:80]}...')
    
    print('\n=== Filtered: severity=critical ===')
    results = chroma_memory.search(query_emb, k=3, where_filter={'severity': 'critical'})
    for r in results:
        print(f'  [{r["metadata"]["severity"]:8s}] [{r["metadata"]["category"]:18s}] {r["text"][:80]}...')
    
    print('\n=== Filtered: category=malware ===')
    results = chroma_memory.search(query_emb, k=3, where_filter={'category': 'malware'})
    for r in results:
        print(f'  [{r["metadata"]["severity"]:8s}] [{r["metadata"]["category"]:18s}] {r["text"][:80]}...')

---
## 6. Threat Pattern Enrichment Pipeline

In [ ]:
class ThreatIntelligenceEngine:
    """
    High-level interface for threat intelligence queries.
    Combines FAISS + ChromaDB + enrichment.
    
    Used by:
    - Deception Agent: to select deception strategy based on known attack patterns
    - Self-Evaluation Agent: to cross-reference detected threats with CTI
    - Orchestration Agent: to provide context for decision making
    """
    
    def __init__(self, faiss_mem: FAISSThreatMemory, embedder: CTIEmbedder,
                 chroma_mem: Optional[ChromaThreatMemory] = None):
        self.faiss_mem = faiss_mem
        self.embedder = embedder
        self.chroma_mem = chroma_mem
    
    def query_threat_context(self, description: str, k: int = 5) -> Dict:
        """
        Given a threat description, retrieve relevant CTI context.
        Returns enriched context for agent decision-making.
        """
        # Get similar threats from FAISS
        faiss_results = self.faiss_mem.search_by_text(
            description, self.embedder, k=k
        )
        
        # Analyze patterns
        categories = Counter()
        severities = Counter()
        all_tags = Counter()
        
        similar_threats = []
        for record, score in faiss_results:
            categories[record.category] += 1
            severities[record.severity] += 1
            for tag in record.tags:
                all_tags[tag] += 1
            similar_threats.append({
                'text': record.text,
                'category': record.category,
                'severity': record.severity,
                'similarity': score,
                'tags': record.tags,
            })
        
        # Determine likely threat type and recommended actions
        likely_category = categories.most_common(1)[0][0] if categories else 'unknown'
        likely_severity = severities.most_common(1)[0][0] if severities else 'medium'
        top_tags = [tag for tag, _ in all_tags.most_common(5)]
        
        # Generate recommendations based on threat type
        recommendations = self._get_recommendations(likely_category, likely_severity)
        
        return {
            'query': description,
            'likely_category': likely_category,
            'likely_severity': likely_severity,
            'confidence': faiss_results[0][1] if faiss_results else 0.0,
            'top_tags': top_tags,
            'similar_threats': similar_threats,
            'recommendations': recommendations,
            'n_matches': len(faiss_results),
        }
    
    def _get_recommendations(self, category: str, severity: str) -> List[str]:
        """Generate defense recommendations based on threat type."""
        recs = {
            'ddos': [
                'Deploy DDoS mitigation / rate limiting',
                'Activate traffic scrubbing',
                'Scale infrastructure capacity',
                'Deploy honeypot to absorb traffic',
            ],
            'malware': [
                'Isolate affected systems immediately',
                'Deploy deceptive file shares (canary files)',
                'Update endpoint detection signatures',
                'Scan network for lateral movement indicators',
            ],
            'phishing': [
                'Alert users and block sender domain',
                'Deploy email-based honeytokens',
                'Rotate potentially compromised credentials',
                'Monitor for credential abuse',
            ],
            'brute_force': [
                'Deploy SSH/RDP honeypots to deflect attacks',
                'Implement progressive lockout',
                'Enable MFA on targeted services',
                'Block source IPs or rate-limit authentication',
            ],
            'injection': [
                'Deploy WAF rules for injection patterns',
                'Isolate vulnerable application',
                'Deploy decoy database with fake data',
                'Patch vulnerable endpoints immediately',
            ],
            'apt': [
                'Full network sweep for indicators of compromise',
                'Deploy advanced honeypots mimicking critical assets',
                'Increase monitoring to maximum level',
                'Engage incident response team',
            ],
            'botnet': [
                'Identify and isolate compromised hosts',
                'Block C2 communication channels',
                'Deploy network-level deception to disrupt C2',
                'Scan for additional compromised systems',
            ],
            'lateral_movement': [
                'Deploy deceptive credentials (honeytokens)',
                'Increase internal network monitoring',
                'Segment network to limit movement',
                'Deploy internal honeypots on likely pivot paths',
            ],
            'exfiltration': [
                'Monitor and block suspicious outbound traffic',
                'Deploy data canaries in sensitive locations',
                'Activate DLP controls',
                'Redirect to honeypot with decoy data',
            ],
        }
        
        base_recs = recs.get(category, ['Increase monitoring', 'Deploy generic honeypots', 'Alert security team'])
        
        if severity in ('critical', 'high'):
            base_recs.insert(0, 'PRIORITY: Immediate incident response escalation')
        
        return base_recs
    
    def get_attack_profile(self, attack_indicators: List[str]) -> Dict:
        """
        Given multiple indicators, build a comprehensive attack profile.
        Used by the orchestration agent to understand the full attack picture.
        """
        all_categories = Counter()
        all_severities = Counter()
        all_recs = []
        
        for indicator in attack_indicators:
            ctx = self.query_threat_context(indicator, k=3)
            all_categories[ctx['likely_category']] += 1
            all_severities[ctx['likely_severity']] += 1
            all_recs.extend(ctx['recommendations'])
        
        # Deduplicate recommendations
        unique_recs = list(dict.fromkeys(all_recs))
        
        return {
            'n_indicators': len(attack_indicators),
            'attack_categories': dict(all_categories.most_common()),
            'overall_severity': all_severities.most_common(1)[0][0] if all_severities else 'medium',
            'recommendations': unique_recs[:10],
        }


# Create engine
intel_engine = ThreatIntelligenceEngine(faiss_memory, embedder, chroma_memory)
print('ThreatIntelligenceEngine ready.')

In [ ]:
# Test the TI engine
test_scenarios = [
    'Unusual outbound DNS traffic with large TXT records detected from database server',
    'Multiple failed SSH login attempts from rotating IP addresses on critical server',
    'Suspicious PowerShell execution with base64 encoded commands on workstation',
]

for scenario in test_scenarios:
    print(f'\n{"="*80}')
    print(f'Scenario: {scenario}')
    print(f'{"="*80}')
    
    ctx = intel_engine.query_threat_context(scenario)
    print(f'  Category:    {ctx["likely_category"]}')
    print(f'  Severity:    {ctx["likely_severity"]}')
    print(f'  Confidence:  {ctx["confidence"]:.4f}')
    print(f'  Tags:        {ctx["top_tags"]}')
    print(f'  Recommendations:')
    for rec in ctx['recommendations'][:4]:
        print(f'    - {rec}')

In [ ]:
# Test multi-indicator attack profiling
attack_indicators = [
    'Port scan on multiple hosts detected from external IP',
    'Brute force SSH login attempt on web server succeeded',
    'Lateral movement detected via SMB from compromised web server to database server',
    'Large data transfer to external IP from database server on unusual port',
]

print('=== Multi-Indicator Attack Profile ===')
profile = intel_engine.get_attack_profile(attack_indicators)
print(f'Indicators analyzed: {profile["n_indicators"]}')
print(f'Attack categories: {profile["attack_categories"]}')
print(f'Overall severity: {profile["overall_severity"]}')
print(f'Recommendations:')
for rec in profile['recommendations']:
    print(f'  - {rec}')

---
## 7. Embedding Visualization

In [ ]:
from sklearn.manifold import TSNE

# t-SNE visualization of threat embeddings
emb_matrix = np.vstack([r.embedding for r in all_records if r.embedding is not None])
categories_list = [r.category for r in all_records if r.embedding is not None]

tsne = TSNE(n_components=2, random_state=42, perplexity=min(15, len(all_records)-1))
emb_2d = tsne.fit_transform(emb_matrix)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
unique_cats = list(set(categories_list))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
cat_color = {cat: colors[i] for i, cat in enumerate(unique_cats)}

for cat in unique_cats:
    mask = np.array([c == cat for c in categories_list])
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
               label=cat, alpha=0.7, s=60, color=cat_color[cat], edgecolors='white', linewidth=0.5)

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_title('Threat Intelligence Embedding Space (t-SNE)', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## 8. Save & Export

In [ ]:
# Save FAISS index
faiss_memory.save(MODELS_DIR / 'faiss')

# Save embedder config
embedder_config = {
    'model_name': embedder.model_name,
    'embed_dim': embedder.embed_dim,
    'fallback': embedder.fallback,
    'n_records': len(all_records),
}
with open(MODELS_DIR / 'embedder_config.json', 'w') as f:
    json.dump(embedder_config, f, indent=2)

print(f'All artifacts saved to {MODELS_DIR}')
print(f'Files:')
for f in sorted(MODELS_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(MODELS_DIR)}')

In [ ]:
def load_threat_intel_engine(models_dir=None):
    """
    Load the threat intelligence engine for use in other notebooks.
    Used by Notebook 08, 09, and 10.
    """
    if models_dir is None:
        models_dir = Path('../models/threat_intel')
    else:
        models_dir = Path(models_dir)
    
    # Load embedder config
    with open(models_dir / 'embedder_config.json', 'r') as f:
        config = json.load(f)
    
    embedder = CTIEmbedder(config['model_name'])
    faiss_mem = FAISSThreatMemory.load(models_dir / 'faiss', embedder)
    
    # Try loading ChromaDB
    chroma = None
    chroma_path = models_dir / 'chromadb'
    if chroma_path.exists():
        chroma = ChromaThreatMemory(
            collection_name='acdada_threat_intel',
            persist_dir=str(chroma_path),
        )
    
    engine = ThreatIntelligenceEngine(faiss_mem, embedder, chroma)
    return engine


# Test load
loaded_engine = load_threat_intel_engine()
test = loaded_engine.query_threat_context('ransomware attack', k=2)
print(f'Loaded engine test — Category: {test["likely_category"]}, Severity: {test["likely_severity"]}')
print('\n✓ Notebook 07 complete. Ready for Notebook 08 (Self-Evaluation Agent).')